In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/colab_upload'  # matches the folder you uploaded

import sys
sys.path.insert(0, PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)


Mounted at /content/drive


In [2]:
!pip install -q dagshub mlflow optuna statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11

In [4]:
import json
import numpy as np
import pandas as pd
import torch
import dagshub
import mlflow

from utils import nbeats_plots as P
from utils.nbeats_data import build_eval_window, build_panel
from utils.nbeats_ensemble import (
    ensemble_rolling_forecast, rolling_block_forecast, train_final_member, compute_wmae,
)
from utils.nbeats_train import HORIZON, local_split_idx, make_cv_folds, run_cv_for_config

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE, '--', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'no GPU found!')

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'
PLOTS_DIR = 'plots'
REPORTS_DIR = 'reports'
MODELS_DIR = 'models'
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

EVALUABLE_MULTIPLIERS = [2, 3, 4, 5, 6]
PRODUCTION_MULTIPLIERS = [2, 3, 4, 5, 6, 7]

dagshub.init(repo_owner='tgela23', repo_name='ml-final-project', mlflow=True)
mlflow.set_experiment('NBEATS_Training')

Using device: cuda -- Tesla T4


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=72278fe8-c80d-4846-b2bb-d0aa64ec7d00&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=209b01ccb995c7952323cd612bd689541f3fcda8215773276fe718cc355cb779




Accessing as tgura23

Initialized MLflow to track repo "tgela23/ml-final-project"

Repository tgela23/ml-final-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/1c9cdd4bf8c7403c923486b25fe1c8b0', creation_time=1784729736800, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784729736800, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

In [5]:
BEST_GENERIC_CFG = dict(
    architecture='generic', n_stacks=2, n_blocks=2, layer_size=256, n_fc_layers=4,
    lookback_multiplier=4, loss='smape', batch_size=256, optimizer='adam',
    learning_rate=0.0007279854007274201, weight_decay=3.822799078402829e-06,
    dropout=0.19094568604860213, share_weights=False, trend_degree=3,
)
BEST_GENERIC_TRIAL_NUM = 23
BEST_GENERIC_CV_WMAE = 1911.095445291105

BEST_INTERPRETABLE_CFG = dict(
    architecture='interpretable', n_blocks=2, layer_size=512, n_fc_layers=5,
    lookback_multiplier=4, loss='mae', batch_size=1024, optimizer='adam',
    learning_rate=0.0010157001316706408, weight_decay=5.973412665154223e-06,
    dropout=0.18797923060380148, share_weights=True, trend_degree=4,
)
BEST_INTERPRETABLE_TRIAL_NUM = 4
BEST_INTERPRETABLE_CV_WMAE = 1902.3813188070403

In [6]:
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
sales, store_ids, dept_ids, store_types, all_dates, is_holiday = build_panel(train, stores)
n_dates = sales.shape[1]
local_train_end_idx = local_split_idx(n_dates)
cv_folds = make_cv_folds(n_dates)


def series_label(s):
    return f'Store {store_ids[s]} Dept {dept_ids[s]}'


def per_series_wmae(y_true, y_pred, holiday_grid, series_idx):
    abs_err = np.abs(y_true - y_pred)
    weights = np.where(holiday_grid, 5, 1)
    out = {}
    for s in np.unique(series_idx):
        mask = series_idx == s
        w = weights[mask]
        out[s] = float((abs_err[mask] * w).sum() / w.sum())
    return out


def breakdown_wmae(y_true, y_pred, holiday_grid, series_idx, store_types):
    holiday_mask = holiday_grid.astype(bool)
    overall = compute_wmae(y_true, y_pred, holiday_grid)
    holiday_wmae = compute_wmae(y_true[holiday_mask], y_pred[holiday_mask], holiday_grid[holiday_mask]) \
        if holiday_mask.any() else float('nan')
    non_holiday_wmae = compute_wmae(y_true[~holiday_mask], y_pred[~holiday_mask], holiday_grid[~holiday_mask])
    type_wmae = {}
    for t in ['A', 'B', 'C']:
        type_series = set(np.where(store_types == t)[0])
        mask = np.array([s in type_series for s in series_idx])
        if mask.sum() == 0:
            continue
        type_wmae[t] = compute_wmae(y_true[mask], y_pred[mask], holiday_grid[mask])
    return overall, holiday_wmae, non_holiday_wmae, type_wmae


def loss_curve_and_epochs(cfg, max_epochs=60, patience=8):
    result = run_cv_for_config(cfg, sales, is_holiday, cv_folds[-1:], max_epochs=max_epochs,
                                patience=patience, min_folds=1, device=DEVICE)
    if result is None:
        return {'train_loss': [], 'val_wmae': []}, max_epochs // 2
    fd = result['fold_details'][0]
    return fd['history'], max(1, int(round(fd['best_epoch'])))


results = {}

In [7]:
cfg = dict(BEST_GENERIC_CFG)
print(f'Best generic config (trial {BEST_GENERIC_TRIAL_NUM}, CV mean_wmae={BEST_GENERIC_CV_WMAE:.2f}): {cfg}')
with mlflow.start_run(run_name='NBEATS_Best_Generic_Colab'):
    mlflow.log_params(cfg)
    mlflow.log_metric('cv_mean_wmae', BEST_GENERIC_CV_WMAE)
    mlflow.log_param('device', DEVICE)

    curve_hist, n_epochs = loss_curve_and_epochs(cfg)
    mlflow.log_metric('final_fit_n_epochs', n_epochs)
    for ep, (tl, vw) in enumerate(zip(curve_hist['train_loss'], curve_hist['val_wmae']), start=1):
        mlflow.log_metric('train_loss', tl, step=ep)
        mlflow.log_metric('val_wmae', vw, step=ep)

    model_g, hist_g, lb_g = train_final_member(cfg, sales, is_holiday, local_train_end_idx, n_epochs, device=DEVICE)

    y_true, y_pred, hgrid, sidx, block, step = rolling_block_forecast(
        model_g, lb_g, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)

    P.plot_loss_curves(curve_hist, 'Best generic (Colab GPU) -- train/val curves', f'{PLOTS_DIR}/nbeats_loss_curves_generic_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_loss_curves_generic_colab.png')
    torch.save(model_g.state_dict(), f'{MODELS_DIR}/nbeats_best_generic_colab.pt')
    mlflow.log_artifact(f'{MODELS_DIR}/nbeats_best_generic_colab.pt')

    print(f'  local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')
    results['best_generic'] = dict(
        trial=BEST_GENERIC_TRIAL_NUM, config=cfg, cv_mean_wmae=BEST_GENERIC_CV_WMAE,
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )


Best generic config (trial 23, CV mean_wmae=1911.10): {'architecture': 'generic', 'n_stacks': 2, 'n_blocks': 2, 'layer_size': 256, 'n_fc_layers': 4, 'lookback_multiplier': 4, 'loss': 'smape', 'batch_size': 256, 'optimizer': 'adam', 'learning_rate': 0.0007279854007274201, 'weight_decay': 3.822799078402829e-06, 'dropout': 0.19094568604860213, 'share_weights': False, 'trend_degree': 3}
  local_test WMAE: overall=2161.25 holiday=3134.43 non_holiday=1756.13 by_type={'A': np.float64(2495.684571692365), 'B': np.float64(2003.3678912107828), 'C': np.float64(910.3641942846579)}
🏃 View run NBEATS_Best_Generic_Colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/e9f00aa1857a4ea493148e53b8766412
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1


In [12]:
import torch
import numpy as np

def plot_decomposition_fixed(model, X_series, dates, series_label, path):
    from utils.nbeats_data import scale_windows
    device = next(model.parameters()).device
    scale = scale_windows(X_series[None, :])
    x = torch.tensor(X_series[None, :] / scale, dtype=torch.float32, device=device)
    with torch.no_grad():
        forecast, backcast_resid, stack_forecasts = model(x, return_decomposition=True)
    keys = list(stack_forecasts.keys())
    trend_key = next((k for k in keys if 'trend' in k), None)
    season_key = next((k for k in keys if 'seasonality' in k), None)

    fig, ax = P.plt.subplots(figsize=(12, 5))
    horizon = forecast.shape[1]
    lookback = X_series.shape[0]
    t_hist = np.arange(-lookback, 0)
    t_fut = np.arange(0, horizon)

    ax.plot(t_hist, X_series, color=P.ACTUAL_COLOR, label='Actual (lookback window)')
    ax.plot(t_fut, forecast[0].cpu().numpy() * scale[0], color=P.FORECAST_COLOR, label='Total forecast')
    if trend_key is not None:
        ax.plot(t_fut, stack_forecasts[trend_key][0].cpu().numpy() * scale[0], color=P.TREND_COLOR,
                linestyle='--', label='Trend component')
    if season_key is not None:
        ax.plot(t_fut, stack_forecasts[season_key][0].cpu().numpy() * scale[0], color=P.SEASON_COLOR,
                linestyle='--', label='Seasonality component')
    ax.axvline(0, color='gray', linewidth=1, linestyle=':')
    ax.set_title(f'N-BEATS interpretable decomposition — {series_label}')
    ax.set_xlabel('Weeks relative to forecast origin')
    ax.set_ylabel('Weekly_Sales')
    ax.legend()
    P._savefig(fig, path)

def plot_backcast_reconstruction_fixed(model, X_series, series_label, path):
    from utils.nbeats_data import scale_windows
    device = next(model.parameters()).device
    scale = scale_windows(X_series[None, :])
    x = torch.tensor(X_series[None, :] / scale, dtype=torch.float32, device=device)
    with torch.no_grad():
        _, backcast_residual = model(x)
    reconstructed = X_series - backcast_residual[0].cpu().numpy() * scale[0]

    fig, ax = P.plt.subplots(figsize=(12, 5))
    t = np.arange(len(X_series))
    ax.plot(t, X_series, color=P.ACTUAL_COLOR, label='Actual input window')
    ax.plot(t, reconstructed, color=P.FORECAST_COLOR, linestyle='--', label='Model backcast reconstruction')
    ax.set_title(f'Backcast reconstruction — {series_label}')
    ax.set_xlabel('Week (within lookback window)')
    ax.set_ylabel('Weekly_Sales')
    ax.legend()
    P._savefig(fig, path)

P.plot_decomposition = plot_decomposition_fixed
P.plot_backcast_reconstruction = plot_backcast_reconstruction_fixed
print('Patched plot_decomposition / plot_backcast_reconstruction for GPU')

Patched plot_decomposition / plot_backcast_reconstruction for GPU


In [10]:

import importlib
import utils.nbeats_plots
importlib.reload(utils.nbeats_plots)
P = utils.nbeats_plots

In [13]:
cfg = dict(BEST_INTERPRETABLE_CFG)
print(f'Best interpretable config (trial {BEST_INTERPRETABLE_TRIAL_NUM}, CV mean_wmae={BEST_INTERPRETABLE_CV_WMAE:.2f}): {cfg}')
with mlflow.start_run(run_name='NBEATS_Best_Interpretable_Colab'):
    mlflow.log_params(cfg)
    mlflow.log_metric('cv_mean_wmae', BEST_INTERPRETABLE_CV_WMAE)
    mlflow.log_param('device', DEVICE)

    curve_hist_i, n_epochs_i = loss_curve_and_epochs(cfg)
    mlflow.log_metric('final_fit_n_epochs', n_epochs_i)
    for ep, (tl, vw) in enumerate(zip(curve_hist_i['train_loss'], curve_hist_i['val_wmae']), start=1):
        mlflow.log_metric('train_loss', tl, step=ep)
        mlflow.log_metric('val_wmae', vw, step=ep)

    model_i, hist_i, lb_i = train_final_member(cfg, sales, is_holiday, local_train_end_idx, n_epochs_i, device=DEVICE)

    y_true, y_pred, hgrid, sidx, block, step = rolling_block_forecast(
        model_i, lb_i, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)

    P.plot_loss_curves(curve_hist_i, 'Best interpretable (Colab GPU) -- train/val curves', f'{PLOTS_DIR}/nbeats_loss_curves_interpretable_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_loss_curves_interpretable_colab.png')

    Xe, series_idx_i = build_eval_window(sales, lb_i, local_train_end_idx)
    P.plot_decomposition(model_i, Xe[0], all_dates, series_label(series_idx_i[0]), f'{PLOTS_DIR}/nbeats_decomposition_colab.png')
    P.plot_backcast_reconstruction(model_i, Xe[0], series_label(series_idx_i[0]), f'{PLOTS_DIR}/nbeats_backcast_reconstruction_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_decomposition_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_backcast_reconstruction_colab.png')

    torch.save(model_i.state_dict(), f'{MODELS_DIR}/nbeats_best_interpretable_colab.pt')
    mlflow.log_artifact(f'{MODELS_DIR}/nbeats_best_interpretable_colab.pt')

    print(f'  local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')
    results['best_interpretable'] = dict(
        trial=BEST_INTERPRETABLE_TRIAL_NUM, config=cfg, cv_mean_wmae=BEST_INTERPRETABLE_CV_WMAE,
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )

Best interpretable config (trial 4, CV mean_wmae=1902.38): {'architecture': 'interpretable', 'n_blocks': 2, 'layer_size': 512, 'n_fc_layers': 5, 'lookback_multiplier': 4, 'loss': 'mae', 'batch_size': 1024, 'optimizer': 'adam', 'learning_rate': 0.0010157001316706408, 'weight_decay': 5.973412665154223e-06, 'dropout': 0.18797923060380148, 'share_weights': True, 'trend_degree': 4}
  local_test WMAE: overall=2286.51 holiday=3352.35 non_holiday=1842.81 by_type={'A': np.float64(2646.1527168860102), 'B': np.float64(2113.3346235037384), 'C': np.float64(955.6731296576778)}
🏃 View run NBEATS_Best_Interpretable_Colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/30b633ead48e454fb5f242e0667dae13
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1


In [14]:
base_configs = [BEST_GENERIC_CFG, BEST_INTERPRETABLE_CFG]
print(f'Training evaluable ensemble: multipliers {EVALUABLE_MULTIPLIERS} x {len(base_configs)} base config(s)')
with mlflow.start_run(run_name='NBEATS_Ensemble_Evaluable_Colab'):
    mlflow.log_param('multipliers', EVALUABLE_MULTIPLIERS)
    mlflow.log_param('n_base_configs', len(base_configs))
    mlflow.log_param('device', DEVICE)
    members = []
    for base_cfg in base_configs:
        _, base_n_epochs = loss_curve_and_epochs(base_cfg)
        for mult in EVALUABLE_MULTIPLIERS:
            cfg = dict(base_cfg, lookback_multiplier=mult)
            with mlflow.start_run(run_name=f'member_{base_cfg["architecture"]}_{mult}x_colab', nested=True):
                mlflow.log_params(cfg)
                mlflow.log_metric('n_epochs', base_n_epochs)
                model_m, hist_m, lb_m = train_final_member(
                    cfg, sales, is_holiday, local_train_end_idx, base_n_epochs, device=DEVICE)
                print(f'  member {base_cfg["architecture"]} x{mult}: trained {len(hist_m["train_loss"])} epochs, '
                      f'final train_loss={hist_m["train_loss"][-1]:.2f}')
                members.append({'model': model_m, 'lookback': lb_m, 'config': cfg})

    y_true, y_pred, hgrid, sidx, block, step = ensemble_rolling_forecast(
        members, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)
    print(f'  ENSEMBLE local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')

    abs_err = np.abs(y_true - y_pred)
    wmae_by_series = per_series_wmae(y_true, y_pred, hgrid, sidx)
    series_wmae_arr = np.array(list(wmae_by_series.values()))
    series_keys_arr = np.array(list(wmae_by_series.keys()))
    order = np.argsort(series_wmae_arr)
    best_s, med_s, worst_s = series_keys_arr[order[0]], series_keys_arr[order[len(order) // 2]], series_keys_arr[order[-1]]

    examples = []
    for label, s in [('Best', best_s), ('Median', med_s), ('Worst', worst_s)]:
        mask = sidx == s
        examples.append((f'{label}: {series_label(s)}', y_true[mask].flatten(), y_pred[mask].flatten(), wmae_by_series[s]))
    P.plot_forecast_vs_actual(examples, f'{PLOTS_DIR}/nbeats_forecast_vs_actual_colab.png')
    P.plot_error_by_horizon(step, abs_err, f'{PLOTS_DIR}/nbeats_error_by_horizon_colab.png')
    P.plot_wmae_distribution(series_wmae_arr, f'{PLOTS_DIR}/nbeats_wmae_distribution_colab.png')
    P.plot_wmae_breakdown(hol, nonhol, by_type, f'{PLOTS_DIR}/nbeats_wmae_breakdown_colab.png')

    row_dates = np.array([
        all_dates[local_train_end_idx + 1 + b * HORIZON: local_train_end_idx + 1 + b * HORIZON + HORIZON]
        for b in block
    ])
    resid = y_true - y_pred
    mean_resid_by_date = pd.Series(resid.flatten(), index=row_dates.flatten()).groupby(level=0).mean().sort_index()
    P.plot_residual_diagnostics(mean_resid_by_date.values, mean_resid_by_date.index, f'{PLOTS_DIR}/nbeats_residuals_colab')

    for fname in ['nbeats_forecast_vs_actual_colab.png', 'nbeats_error_by_horizon_colab.png',
                  'nbeats_wmae_distribution_colab.png', 'nbeats_wmae_breakdown_colab.png',
                  'nbeats_residuals_colab_timeseries.png', 'nbeats_residuals_colab_hist.png', 'nbeats_residuals_colab_acf.png']:
        mlflow.log_artifact(f'{PLOTS_DIR}/{fname}')

    results['ensemble_evaluable'] = dict(
        multipliers=EVALUABLE_MULTIPLIERS, n_members=len(members),
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )


Training evaluable ensemble: multipliers [2, 3, 4, 5, 6] x 2 base config(s)
  member generic x2: trained 19 epochs, final train_loss=0.21
🏃 View run member_generic_2x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/fb54022bcd47409c8da184df0a443221
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x3: trained 19 epochs, final train_loss=0.17
🏃 View run member_generic_3x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/8b4e8428727f46e7ace1e406925f36c0
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x4: trained 19 epochs, final train_loss=0.13
🏃 View run member_generic_4x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/283301979b5842c796d411f790b7ef1e
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x5: trained 19 e

In [15]:
print(f'Training production ensemble: multipliers {PRODUCTION_MULTIPLIERS} (full history, no held-out eval)')
with mlflow.start_run(run_name='NBEATS_Production_Final_Colab'):
    mlflow.log_param('multipliers', PRODUCTION_MULTIPLIERS)
    mlflow.log_param('device', DEVICE)
    prod_cfg = base_configs[0]
    _, prod_n_epochs = loss_curve_and_epochs(prod_cfg)
    mlflow.log_metric('n_epochs', prod_n_epochs)
    for mult in PRODUCTION_MULTIPLIERS:
        cfg = dict(prod_cfg, lookback_multiplier=mult)
        with mlflow.start_run(run_name=f'prod_member_{mult}x_colab', nested=True):
            mlflow.log_params(cfg)
            model_p, hist_p, lb_p = train_final_member(
                cfg, sales, is_holiday, n_dates - 1, prod_n_epochs, device=DEVICE)
            torch.save(model_p.state_dict(), f'{MODELS_DIR}/nbeats_production_{mult}x_colab.pt')
            mlflow.log_artifact(f'{MODELS_DIR}/nbeats_production_{mult}x_colab.pt')
            print(f'  production member x{mult}: trained {len(hist_p["train_loss"])} epochs, '
                  f'final train_loss={hist_p["train_loss"][-1]:.2f}')
    results['production_ensemble'] = dict(multipliers=PRODUCTION_MULTIPLIERS, base_config=prod_cfg)

Training production ensemble: multipliers [2, 3, 4, 5, 6, 7] (full history, no held-out eval)
  production member x2: trained 19 epochs, final train_loss=0.20
🏃 View run prod_member_2x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/10313fbf72274c31bde66ce31a463cbf
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production member x3: trained 19 epochs, final train_loss=0.18
🏃 View run prod_member_3x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/211518c4e6584777b10248de6f13ea37
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production member x4: trained 19 epochs, final train_loss=0.14
🏃 View run prod_member_4x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/436dbf42776444d99323f7273ca66d74
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production me

In [16]:
with open(f'{REPORTS_DIR}/nbeats_finalize_results_colab.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Done. Results written to reports/nbeats_finalize_results_colab.json')

Done. Results written to reports/nbeats_finalize_results_colab.json
